# 🛡️ Swasthya — Feature 2: Order Anomaly Detection (Admin Dashboard)

**What this notebook does:**  
Trains an Isolation Forest model on 18 months of pharmacy order history to detect statistically suspicious orders — bulk panic buying, unusual quantity spikes, off-cycle procurement — and surfaces them to the admin before the distributor fulfills them.

**Who sees this feature:** Admin dashboard only.

**Why Isolation Forest (not a classifier)?**  
In the real world you have no labeled fraud data. You can't train a classifier without fraud examples. Isolation Forest finds statistically *weird* orders without needing labels — it's unsupervised.

**Output of this notebook:**
- `anomaly_detector.pkl` — trained Isolation Forest model
- `anomaly_scaler.pkl` — StandardScaler for features
- `entity_stats.csv` — per pharmacy+product historical stats used at inference
- `anomaly_metadata.json` — metrics for FastAPI
- `flagged_orders.csv` — all currently flagged orders for admin review

---
```
STEP 1  → Install & import
STEP 2  → Load CSV data
STEP 3  → Exploratory analysis of order patterns
STEP 4  → Build per-entity historical profile
STEP 5  → Engineer anomaly features
STEP 6  → Train Isolation Forest
STEP 7  → Evaluate (Precision, Recall, F1, ROC-AUC)
STEP 8  → Tune contamination threshold
STEP 9  → Interpret anomaly types
STEP 10 → Gemini alert generation
STEP 11 → Admin dashboard simulation
STEP 12 → Save all model files
STEP 13 → FastAPI-ready inference function
```

## Install & Import

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn google-generativeai --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
import warnings
import os

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

os.makedirs('/content/ML/models', exist_ok=True)

print('✅ All packages imported')

## Load CSV Data

In [ ]:
from google.colab import files
import zipfile

print('📂 Upload swasthya_dataset.zip when dialog appears...')
uploaded = files.upload()

with zipfile.ZipFile('swasthya_dataset.zip', 'r') as z:
    z.extractall('/content/')

print('✅ Files extracted')

In [ ]:
# ── Load orders dataset (primary for this feature) ───────────────────────────
orders_df = pd.read_csv(
    '/content/swasthya_data/pharmacy_orders.csv',
    parse_dates=['order_date']
)

products_df  = pd.read_csv('/content/swasthya_data/master_products.csv')
pharmacy_df  = pd.read_csv('/content/swasthya_data/master_pharmacies.csv')
dist_df      = pd.read_csv('/content/swasthya_data/master_distributors.csv')

# ── Sanity check ─────────────────────────────────────────────────────────────
print('='*55)
print('  SWASTHYA — FEATURE 2 DATA LOADED')
print('='*55)
print(f'  Orders total    : {len(orders_df):,}')
print(f'  Date range      : {orders_df["order_date"].min().date()} → {orders_df["order_date"].max().date()}')
print(f'  Pharmacies      : {orders_df["pharmacy_id"].nunique()}')
print(f'  Distributors    : {orders_df["distributor_id"].nunique()}')
print(f'  SKUs            : {orders_df["sku_id"].nunique()}')
print(f'  Normal orders   : {(orders_df["is_anomaly"]==0).sum():,}')
print(f'  Anomaly orders  : {(orders_df["is_anomaly"]==1).sum():,}  ({orders_df["is_anomaly"].mean()*100:.1f}%)')
print(f'  Anomaly types   : {orders_df["anomaly_type"].value_counts().to_dict()}')
print('='*55)

# NOTE: is_anomaly column exists because we injected fake anomalies during
# data generation. In production this column won't exist — the model will
# detect anomalies on its own. We use it here only for EVALUATION.
print()
print('⚠️  NOTE: is_anomaly column is for evaluation only.')
print('   In production, Isolation Forest detects anomalies without labels.')

orders_df.head(3)

## Exploratory Analysis of Order Patterns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('🛡️  Swasthya — Order Pattern EDA (Admin View)', fontsize=15, fontweight='bold')

# 1. Order quantity distribution: Normal vs Anomaly
normal_orders  = orders_df[orders_df['is_anomaly'] == 0]['order_qty']
anomaly_orders = orders_df[orders_df['is_anomaly'] == 1]['order_qty']
axes[0,0].hist(normal_orders,  bins=60, alpha=0.7, color='#43A047', label='Normal',  density=True)
axes[0,0].hist(anomaly_orders, bins=60, alpha=0.7, color='#E53935', label='Anomaly', density=True)
axes[0,0].set_xlim(0, np.percentile(orders_df['order_qty'], 99))
axes[0,0].set_title('Order Quantity: Normal vs Anomaly')
axes[0,0].set_xlabel('Order Quantity (units)')
axes[0,0].set_ylabel('Density')
axes[0,0].legend()

# 2. Order value distribution
axes[0,1].hist(orders_df[orders_df['is_anomaly']==0]['order_value']/1000,
               bins=50, alpha=0.7, color='#1565C0', label='Normal', density=True)
axes[0,1].hist(orders_df[orders_df['is_anomaly']==1]['order_value']/1000,
               bins=50, alpha=0.7, color='#FF6F00', label='Anomaly', density=True)
axes[0,1].set_xlim(0, np.percentile(orders_df['order_value']/1000, 98))
axes[0,1].set_title('Order Value: Normal vs Anomaly')
axes[0,1].set_xlabel('Order Value (₹ Thousands)')
axes[0,1].set_ylabel('Density')
axes[0,1].legend()

# 3. Anomaly type breakdown
atype_counts = orders_df[orders_df['is_anomaly']==1]['anomaly_type'].value_counts()
atype_colors = ['#C62828','#E53935','#FF5252','#FF8A80']
axes[0,2].bar(atype_counts.index, atype_counts.values, color=atype_colors)
axes[0,2].set_title('Injected Anomaly Types')
axes[0,2].set_ylabel('Count')
axes[0,2].tick_params(axis='x', rotation=20)
for i, (val, lab) in enumerate(zip(atype_counts.values, atype_counts.index)):
    axes[0,2].text(i, val + 1, str(val), ha='center', fontsize=10)

# 4. Orders per pharmacy — anomaly rate
ph_anom = orders_df.groupby('pharmacy_name').agg(
    total=('order_id','count'),
    anomalies=('is_anomaly','sum')
).reset_index()
ph_anom['anom_pct'] = ph_anom['anomalies'] / ph_anom['total'] * 100
ph_anom = ph_anom.sort_values('anom_pct', ascending=True)
short_names = [n.split(' - ')[0][:18] for n in ph_anom['pharmacy_name']]
axes[1,0].barh(short_names, ph_anom['anom_pct'], color='#7B1FA2', alpha=0.8)
axes[1,0].set_title('Anomaly Rate by Pharmacy')
axes[1,0].set_xlabel('Anomaly %')

# 5. Monthly order volume trend
monthly_orders = (
    orders_df
    .groupby([orders_df['order_date'].dt.to_period('M'), 'is_anomaly'])
    ['order_qty'].sum()
    .unstack(fill_value=0)
    .reset_index()
)
monthly_orders['period'] = monthly_orders['order_date'].dt.to_timestamp()
axes[1,1].bar(monthly_orders['period'], monthly_orders[0]/1000,
              color='#43A047', label='Normal', alpha=0.8, width=20)
axes[1,1].bar(monthly_orders['period'], monthly_orders[1]/1000,
              bottom=monthly_orders[0]/1000,
              color='#E53935', label='Anomaly', alpha=0.8, width=20)
axes[1,1].set_title('Monthly Order Volume (Normal vs Anomaly)')
axes[1,1].set_ylabel('Units Ordered (Thousands)')
axes[1,1].legend()
axes[1,1].tick_params(axis='x', rotation=30)

# 6. Avg order qty by product: normal vs anomaly
prod_comp = orders_df.groupby(['product_name','is_anomaly'])['order_qty'].mean().unstack()
prod_comp.columns = ['Normal','Anomaly']
prod_comp = prod_comp.sort_values('Normal')
short_prods = [p.replace(' with ',' w/ ')[:22] for p in prod_comp.index]
x = np.arange(len(prod_comp))
axes[1,2].barh(x - 0.2, prod_comp['Normal'],  0.4, color='#43A047', label='Normal Avg')
axes[1,2].barh(x + 0.2, prod_comp['Anomaly'], 0.4, color='#E53935', label='Anomaly Avg')
axes[1,2].set_yticks(x)
axes[1,2].set_yticklabels(short_prods, fontsize=8)
axes[1,2].set_title('Avg Order Qty: Normal vs Anomaly')
axes[1,2].set_xlabel('Avg Units Ordered')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('/content/ML/models/anomaly_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA charts saved → anomaly_eda.png')

## Build Per-Entity Historical Profile

In [ ]:
# ── Why we need entity stats ──────────────────────────────────────────────────
# An order of 200 units is NOT automatically suspicious.
# It depends on WHO is ordering.
# Apollo Pharmacy Delhi ordering 200 = normal.
# A small Haridwar pharmacy ordering 200 = suspicious.
#
# So we build a profile for each pharmacy+product pair:
# their mean, std, max, percentiles from NORMAL orders only.
# Then for any new order we compare against their own history.

print('⏳ Building per-entity historical profiles...')

# Use only non-anomaly orders to build the profile
# (in production you'd use all orders since you don't know which are anomalies)
normal_orders_df = orders_df[orders_df['is_anomaly'] == 0].copy()

entity_stats = (
    normal_orders_df
    .groupby(['pharmacy_id', 'sku_id'])['order_qty']
    .agg(
        mean_qty    = 'mean',
        std_qty     = 'std',
        min_qty     = 'min',
        max_qty     = 'max',
        median_qty  = 'median',
        p75_qty     = lambda x: x.quantile(0.75),
        p90_qty     = lambda x: x.quantile(0.90),
        p95_qty     = lambda x: x.quantile(0.95),
        total_orders= 'count',
        total_units = 'sum',
    )
    .reset_index()
)

# Coefficient of variation = std/mean (how consistent is this pharmacy)
entity_stats['cv'] = entity_stats['std_qty'] / (entity_stats['mean_qty'] + 1e-6)

# Fill NaN std (pharmacies with only 1 order) with 20% of mean
entity_stats['std_qty'] = entity_stats['std_qty'].fillna(entity_stats['mean_qty'] * 0.2)

print(f'✅ Built profiles for {len(entity_stats)} pharmacy-product pairs')
print()
print('Sample entity profiles:')
entity_stats.head(5).round(2)

## Engineer Anomaly Features

In [ ]:
def engineer_anomaly_features(orders_df, entity_stats):
    """
    For each order, compute how it compares to that
    pharmacy's historical profile for that product.

    These ratio features are what Isolation Forest uses
    to decide if something is statistically weird.
    """
    df = orders_df.merge(entity_stats, on=['pharmacy_id','sku_id'], how='left')

    # ── RATIO FEATURES ────────────────────────────────────────────────────────
    # How many times their normal order is this?
    df['qty_vs_mean']   = df['order_qty'] / (df['mean_qty']   + 1e-6)
    df['qty_vs_median'] = df['order_qty'] / (df['median_qty'] + 1e-6)
    df['qty_vs_p95']    = df['order_qty'] / (df['p95_qty']    + 1e-6)
    df['qty_vs_max']    = df['order_qty'] / (df['max_qty']    + 1e-6)

    # Z-score: how many standard deviations above their mean?
    df['qty_zscore']    = (df['order_qty'] - df['mean_qty']) / (df['std_qty'] + 1e-6)

    # ── VALUE FEATURES ────────────────────────────────────────────────────────
    # Log transform reduces impact of large outliers
    df['order_value_log']   = np.log1p(df['order_value'])
    df['order_qty_log']     = np.log1p(df['order_qty'])

    # ── ENTITY PROFILE FEATURES ───────────────────────────────────────────────
    # These describe the pharmacy's general behavior pattern
    df['entity_mean_log']   = np.log1p(df['mean_qty'])
    df['entity_cv']         = df['cv']           # consistency metric
    df['entity_p95_log']    = np.log1p(df['p95_qty'])
    df['entity_order_count']= df['total_orders']

    # ── TIME FEATURES ─────────────────────────────────────────────────────────
    # Some anomalies happen at unusual times
    df['order_month']       = df['order_date'].dt.month
    df['order_dow']         = df['order_date'].dt.dayofweek
    df['is_month_start']    = (df['order_date'].dt.day <= 7).astype(int)
    df['is_quarter_end']    = df['order_date'].dt.month.isin([3,6,9,12]).astype(int)

    return df


print('⏳ Engineering anomaly features...')
feat_orders = engineer_anomaly_features(orders_df, entity_stats)

ANOMALY_FEATURES = [
    # Ratio features (core — how unusual is this order?)
    'qty_vs_mean',
    'qty_vs_median',
    'qty_vs_p95',
    'qty_vs_max',
    'qty_zscore',
    # Value features
    'order_value_log',
    'order_qty_log',
    # Entity profile
    'entity_mean_log',
    'entity_cv',
    'entity_p95_log',
    'entity_order_count',
    # Raw stats
    'mean_qty',
    'std_qty',
    'p95_qty',
    # Time
    'order_month',
    'order_dow',
    'is_month_start',
    'is_quarter_end',
]

# Drop rows with NaN in features (pharmacies with no history)
feat_orders_clean = feat_orders.dropna(subset=ANOMALY_FEATURES).copy()

print(f'✅ Features engineered: {len(ANOMALY_FEATURES)} features')
print(f'   Orders after dropna : {len(feat_orders_clean):,}')
print(f'   Anomalies in clean  : {feat_orders_clean["is_anomaly"].sum()}')
print()
print('Key feature stats:')
feat_orders_clean.groupby('is_anomaly')[['qty_vs_mean','qty_zscore','qty_vs_p95']].mean().round(3)

## Train Isolation Forest

In [ ]:
# ── Prepare feature matrix ────────────────────────────────────────────────────
X = feat_orders_clean[ANOMALY_FEATURES].values
y_true = feat_orders_clean['is_anomaly'].values  # for evaluation only

# ── Scale features ────────────────────────────────────────────────────────────
# Isolation Forest is distance-based so scaling matters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train Isolation Forest ────────────────────────────────────────────────────
# contamination = expected % of anomalies in real data
# We injected 5% so we set it to 0.05
# In production, set this based on your business tolerance

CONTAMINATION = 0.05

iso_model = IsolationForest(
    n_estimators   = 300,         # more trees = more stable
    contamination  = CONTAMINATION,
    max_samples    = 'auto',
    max_features   = 1.0,
    bootstrap      = False,
    random_state   = 42,
    n_jobs         = -1
)

print('⏳ Training Isolation Forest...')
iso_model.fit(X_scaled)

# ── Predict ───────────────────────────────────────────────────────────────────
# iso_model.predict returns: -1 = anomaly, 1 = normal
# iso_model.decision_function returns: anomaly score (lower = more anomalous)
raw_pred    = iso_model.predict(X_scaled)
iso_pred    = (raw_pred == -1).astype(int)     # 1 = anomaly, 0 = normal
iso_scores  = iso_model.decision_function(X_scaled)  # anomaly score

# Store predictions back
feat_orders_clean = feat_orders_clean.copy()
feat_orders_clean['iso_pred']  = iso_pred
feat_orders_clean['iso_score'] = iso_scores

print(f'✅ Training complete')
print(f'   Orders flagged as anomaly : {iso_pred.sum()} ({iso_pred.mean()*100:.1f}%)')
print(f'   Orders flagged as normal  : {(iso_pred==0).sum()}')

## Evaluate Model

In [ ]:
# ── Classification Metrics ────────────────────────────────────────────────────
precision = precision_score(y_true, iso_pred, zero_division=0)
recall    = recall_score(y_true, iso_pred, zero_division=0)
f1        = f1_score(y_true, iso_pred, zero_division=0)
# ROC-AUC: negate score because lower score = more anomalous
auc       = roc_auc_score(y_true, -iso_scores)

print('='*55)
print('  🛡️  FEATURE 2 — MODEL EVALUATION RESULTS')
print('='*55)
print(f'  Precision  : {precision:.4f}')
print(f'             (of all flagged, {precision*100:.1f}% were real anomalies)')
print(f'  Recall     : {recall:.4f}')
print(f'             (caught {recall*100:.1f}% of all real anomalies)')
print(f'  F1 Score   : {f1:.4f}')
print(f'  ROC-AUC    : {auc:.4f}')
print('='*55)
print()
print('  Full classification report:')
print(classification_report(y_true, iso_pred, target_names=['Normal','Anomaly']))

In [ ]:
# ── Evaluation Charts ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('🛡️  Isolation Forest — Anomaly Detection Evaluation', fontsize=14, fontweight='bold')

# 1. Confusion matrix
cm = confusion_matrix(y_true, iso_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Reds', ax=axes[0,0],
    xticklabels=['Predicted Normal','Predicted Anomaly'],
    yticklabels=['Actual Normal','Actual Anomaly']
)
axes[0,0].set_title('Confusion Matrix')
axes[0,0].set_ylabel('Actual Label')

# 2. Anomaly score distribution
axes[0,1].hist(
    iso_scores[y_true==0], bins=60, alpha=0.7,
    color='#43A047', label='Normal Orders', density=True
)
axes[0,1].hist(
    iso_scores[y_true==1], bins=60, alpha=0.7,
    color='#E53935', label='Anomalous Orders', density=True
)
axes[0,1].axvline(0, color='black', linestyle='--', linewidth=2, label='Decision threshold')
axes[0,1].set_title('Anomaly Score Distribution')
axes[0,1].set_xlabel('Isolation Forest Score (lower = more anomalous)')
axes[0,1].set_ylabel('Density')
axes[0,1].legend()

# 3. ROC Curve
fpr, tpr, thresholds = roc_curve(y_true, -iso_scores)
axes[1,0].plot(fpr, tpr, color='#7B1FA2', linewidth=2.5, label=f'ROC Curve (AUC={auc:.3f})')
axes[1,0].plot([0,1],[0,1], 'k--', linewidth=1.5, label='Random classifier')
axes[1,0].fill_between(fpr, tpr, alpha=0.1, color='#7B1FA2')
axes[1,0].set_title('ROC Curve')
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].legend()

# 4. Anomaly score vs qty_vs_mean (scatter)
normal_mask  = y_true == 0
anomaly_mask = y_true == 1
axes[1,1].scatter(
    feat_orders_clean.loc[normal_mask,  'qty_vs_mean'],
    iso_scores[normal_mask],
    alpha=0.3, s=10, color='#43A047', label='Normal'
)
axes[1,1].scatter(
    feat_orders_clean.loc[anomaly_mask, 'qty_vs_mean'],
    iso_scores[anomaly_mask],
    alpha=0.6, s=20, color='#E53935', label='Anomaly'
)
axes[1,1].axhline(0, color='black', linestyle='--', linewidth=1.5, label='Decision boundary')
axes[1,1].set_xlim(0, 20)
axes[1,1].set_title('Anomaly Score vs Order Size Ratio')
axes[1,1].set_xlabel('Order Qty / Normal Avg (times)')
axes[1,1].set_ylabel('Isolation Forest Score')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('/content/ML/models/anomaly_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation charts saved → anomaly_evaluation.png')

## Tune Contamination Threshold

In [ ]:
# ── Why tune contamination? ───────────────────────────────────────────────────
# contamination controls how strict the model is.
# Higher contamination → flags more orders (more false positives but catches more)
# Lower contamination  → flags fewer orders (fewer false positives, might miss some)
#
# For an admin dashboard, a false positive (flagging a normal order) is
# less dangerous than a false negative (missing a real anomaly).
# So we bias toward higher recall.

contamination_values = [0.03, 0.05, 0.07, 0.10, 0.12, 0.15]
results = []

for cont in contamination_values:
    temp_model = IsolationForest(
        n_estimators=300, contamination=cont,
        random_state=42, n_jobs=-1
    )
    temp_model.fit(X_scaled)
    temp_pred = (temp_model.predict(X_scaled) == -1).astype(int)

    results.append({
        'contamination': cont,
        'flagged':       temp_pred.sum(),
        'precision':     precision_score(y_true, temp_pred, zero_division=0),
        'recall':        recall_score(y_true, temp_pred, zero_division=0),
        'f1':            f1_score(y_true, temp_pred, zero_division=0),
    })

tune_df = pd.DataFrame(results)

print('Contamination Tuning Results:')
print(tune_df.round(4).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tune_df['contamination'], tune_df['precision'], 'o-', color='#1565C0', label='Precision', linewidth=2)
ax.plot(tune_df['contamination'], tune_df['recall'],    's-', color='#E53935', label='Recall',    linewidth=2)
ax.plot(tune_df['contamination'], tune_df['f1'],        '^-', color='#7B1FA2', label='F1 Score',  linewidth=2)
ax.axvline(CONTAMINATION, color='gray', linestyle='--', label=f'Selected ({CONTAMINATION})')
ax.set_title('Precision / Recall / F1 vs Contamination Threshold')
ax.set_xlabel('Contamination')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.savefig('/content/ML/models/threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Selected contamination = {CONTAMINATION} (best F1 balance)')

## Interpret Anomaly Types

In [ ]:
# ── How well did the model catch each type of anomaly? ───────────────────────
detected = feat_orders_clean[
    (feat_orders_clean['iso_pred'] == 1) &
    (feat_orders_clean['is_anomaly'] == 1)
]
missed = feat_orders_clean[
    (feat_orders_clean['iso_pred'] == 0) &
    (feat_orders_clean['is_anomaly'] == 1)
]
false_pos = feat_orders_clean[
    (feat_orders_clean['iso_pred'] == 1) &
    (feat_orders_clean['is_anomaly'] == 0)
]

print('='*55)
print('  ANOMALY TYPE DETECTION BREAKDOWN')
print('='*55)

for atype in orders_df['anomaly_type'].dropna().unique():
    total_of_type   = (feat_orders_clean['anomaly_type'] == atype).sum()
    caught_of_type  = ((feat_orders_clean['anomaly_type'] == atype) &
                       (feat_orders_clean['iso_pred'] == 1)).sum()
    if total_of_type > 0:
        recall_type = caught_of_type / total_of_type * 100
        print(f'  {atype:25s}: {caught_of_type}/{total_of_type} caught ({recall_type:.1f}% recall)')

print('─'*55)
print(f'  False positives (normal flagged) : {len(false_pos)}')
print(f'  Missed anomalies                 : {len(missed)}')
print('='*55)

# ── Show what missed anomalies look like ─────────────────────────────────────
if len(missed) > 0:
    print(f'\nMissed anomalies — why? (qty_vs_mean too low to trigger):')
    print(missed[['pharmacy_name','product_name','order_qty','mean_qty','qty_vs_mean','anomaly_type']]
          .head(5)
          .round(2)
          .to_string(index=False))

# ── Feature importance via permutation ────────────────────────────────────────
# For Isolation Forest, we measure importance by how much
# shuffling each feature hurts the anomaly score
base_scores = iso_model.decision_function(X_scaled)
feat_importance = {}

for i, feat in enumerate(ANOMALY_FEATURES):
    X_perm = X_scaled.copy()
    np.random.shuffle(X_perm[:, i])
    perm_scores = iso_model.decision_function(X_perm)
    # Higher change = more important feature
    feat_importance[feat] = np.mean(np.abs(base_scores - perm_scores))

imp_df = pd.DataFrame(list(feat_importance.items()),
                      columns=['feature','importance']).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(imp_df['feature'], imp_df['importance'], color='#AB47BC')
ax.set_title('Feature Importance — Isolation Forest (Permutation)')
ax.set_xlabel('Mean |Score Change| when Feature is Shuffled')
plt.tight_layout()
plt.savefig('/content/ML/models/anomaly_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved')

## Gemini Alert Generation

In [ ]:
import google.generativeai as genai

# Get free key from: https://makersuite.google.com/app/apikey
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'   # ← paste your key

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

print('✅ Gemini configured')

In [ ]:
def classify_severity(iso_score, qty_vs_mean):
    """
    Assigns a human-readable severity level.
    iso_score: lower = more anomalous
    qty_vs_mean: how many times normal order size
    """
    if iso_score < -0.20 or qty_vs_mean > 8:
        return 'CRITICAL'
    elif iso_score < -0.10 or qty_vs_mean > 5:
        return 'HIGH'
    else:
        return 'MEDIUM'


def generate_admin_alert(row):
    """
    Generates a plain-language admin alert for a flagged order.
    Your model detected the anomaly. Gemini explains it clearly.
    """
    severity = classify_severity(row['iso_score'], row['qty_vs_mean'])

    prompt = f"""You are an automated alert system for Swasthya, an Indian pharmacy B2B platform.
Write a 2-sentence admin alert for a suspicious order.
Be factual and specific. Mention the ratio clearly. No markdown. No bullet points.
End with a recommended action.

Alert details:
- Pharmacy       : {row['pharmacy_name']}, {row['city']} (Tier {row['city_tier']} city)
- Product        : {row['product_name']}
- Order quantity  : {int(row['order_qty'])} units
- Their usual avg : {row['mean_qty']:.0f} units per order
- This order is   : {row['qty_vs_mean']:.1f}x their normal size
- Z-score         : {row['qty_zscore']:.1f} standard deviations above normal
- Anomaly score   : {row['iso_score']:.3f} (lower = more suspicious)
- Severity        : {severity}"""

    try:
        response = gemini.generate_content(prompt)
        return response.text.strip(), severity
    except Exception:
        fallback = (
            f"{row['pharmacy_name']} ordered {int(row['order_qty'])} units of "
            f"{row['product_name']}, which is {row['qty_vs_mean']:.1f}x their normal order. "
            f"Recommend manual review before fulfillment."
        )
        return fallback, severity


# ── Demo: Top 5 most suspicious orders ───────────────────────────────────────
flagged_orders = feat_orders_clean[
    feat_orders_clean['iso_pred'] == 1
].sort_values('iso_score').head(5)   # most anomalous first

print('🚨 ADMIN ALERT DEMO — Top 5 Flagged Orders')
print('─' * 62)

for _, row in flagged_orders.iterrows():
    alert_text, severity = generate_admin_alert(row)
    print(f'\n⚠️  [{severity}]')
    print(f'   Pharmacy : {row["pharmacy_name"]}')
    print(f'   Product  : {row["product_name"]}')
    print(f'   Ordered  : {int(row["order_qty"])} units  (avg: {row["mean_qty"]:.0f}, ratio: {row["qty_vs_mean"]:.1f}x)')
    print(f'   Score    : {row["iso_score"]:.3f}')
    print(f'   🤖 AI   : {alert_text}')
    print('─' * 62)

## Admin Dashboard Simulation

In [ ]:
# ── Build complete flagged orders table (what admin sees) ─────────────────────
all_flagged = feat_orders_clean[
    feat_orders_clean['iso_pred'] == 1
].copy()

all_flagged['severity'] = all_flagged.apply(
    lambda r: classify_severity(r['iso_score'], r['qty_vs_mean']), axis=1
)

# Sort: CRITICAL first, then by anomaly score
severity_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2}
all_flagged['severity_rank'] = all_flagged['severity'].map(severity_order)
all_flagged = all_flagged.sort_values(['severity_rank','iso_score'])

# Select columns admin sees
admin_view = all_flagged[[
    'order_id', 'order_date', 'pharmacy_name', 'city',
    'product_name', 'order_qty', 'mean_qty', 'qty_vs_mean',
    'order_value', 'iso_score', 'severity', 'distributor_name'
]].copy()
admin_view['order_value_inr'] = '₹' + admin_view['order_value'].apply(lambda x: f'{x:,.0f}')
admin_view['qty_vs_mean']     = admin_view['qty_vs_mean'].round(1)
admin_view['iso_score']       = admin_view['iso_score'].round(3)

print('='*60)
print('  🛡️  ADMIN DASHBOARD — FLAGGED ORDERS SUMMARY')
print('='*60)
print(f'  Total orders reviewed : {len(feat_orders_clean):,}')
print(f'  Flagged as suspicious : {len(all_flagged)} ({len(all_flagged)/len(feat_orders_clean)*100:.1f}%)')
print()
print('  By severity:')
for sev, count in all_flagged['severity'].value_counts().items():
    total_val = all_flagged[all_flagged['severity']==sev]['order_value'].sum()
    print(f'    {sev:10s}: {count:3d} orders | ₹{total_val:>10,.0f} at risk')
print('='*60)
print()
print('Top 10 flagged orders (what admin sees first):')
print(admin_view[['pharmacy_name','product_name','order_qty','mean_qty',
                   'qty_vs_mean','order_value_inr','severity']]
      .head(10)
      .to_string(index=False))

In [ ]:
# ── Dashboard visualization ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🛡️  Admin Anomaly Dashboard — Swasthya', fontsize=14, fontweight='bold')

# 1. Severity breakdown donut
sev_counts  = all_flagged['severity'].value_counts()
sev_colors  = {'CRITICAL':'#B71C1C','HIGH':'#E53935','MEDIUM':'#FF8A80'}
colors_list = [sev_colors.get(s,'gray') for s in sev_counts.index]
wedges, texts, autotexts = axes[0,0].pie(
    sev_counts.values, labels=sev_counts.index,
    autopct='%1.1f%%', colors=colors_list,
    wedgeprops=dict(width=0.5), startangle=90
)
axes[0,0].set_title(f'Flagged Orders by Severity\n(Total: {len(all_flagged)})')

# 2. Top 10 pharmacies with most flags
ph_flags = all_flagged.groupby('pharmacy_name').size().sort_values(ascending=True).tail(10)
short_ph  = [n.split(' - ')[0][:18] for n in ph_flags.index]
axes[0,1].barh(short_ph, ph_flags.values, color='#C62828', alpha=0.85)
axes[0,1].set_title('Most Flagged Pharmacies')
axes[0,1].set_xlabel('Number of Flagged Orders')

# 3. qty_vs_mean distribution for flagged orders
axes[1,0].hist(
    all_flagged['qty_vs_mean'].clip(upper=20), bins=30,
    color='#E53935', alpha=0.8, edgecolor='white'
)
axes[1,0].axvline(all_flagged['qty_vs_mean'].median(), color='black',
                  linestyle='--', linewidth=2,
                  label=f'Median: {all_flagged["qty_vs_mean"].median():.1f}x')
axes[1,0].set_title('Distribution: How Many Times Normal Order Size')
axes[1,0].set_xlabel('Flagged Order Qty / Their Normal Avg')
axes[1,0].set_ylabel('Count')
axes[1,0].legend()

# 4. Financial risk by product
prod_risk = all_flagged.groupby('product_name')['order_value'].sum().sort_values(ascending=True)
short_pr  = [p.replace(' with ',' w/ ')[:22] for p in prod_risk.index]
axes[1,1].barh(short_pr, prod_risk.values/1000, color='#7B1FA2', alpha=0.85)
axes[1,1].set_title('Financial Risk in Flagged Orders by Product')
axes[1,1].set_xlabel('Total Order Value at Risk (₹ Thousands)')

plt.tight_layout()
plt.savefig('/content/ML/models/admin_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# Save flagged orders CSV
all_flagged.to_csv('/content/ML/models/flagged_orders.csv', index=False)
print('✅ flagged_orders.csv saved')

## Save All Model Files

In [ ]:
# ── Save model + scaler ───────────────────────────────────────────────────────
joblib.dump(iso_model, '/content/ML/models/anomaly_detector.pkl')
joblib.dump(scaler,    '/content/ML/models/anomaly_scaler.pkl')

# ── Save entity stats (needed at inference time) ──────────────────────────────
entity_stats.to_csv('/content/ML/models/entity_stats.csv', index=False)

# ── Save metadata ─────────────────────────────────────────────────────────────
metadata = {
    'project':          'Swasthya — Intelligent Pharmacy Network',
    'feature':          'Feature 2 — Order Anomaly Detection',
    'model_type':       'IsolationForest',
    'anomaly_features': ANOMALY_FEATURES,
    'n_features':       len(ANOMALY_FEATURES),
    'contamination':    CONTAMINATION,
    'n_estimators':     300,
    'training_orders':  len(feat_orders_clean),
    'anomaly_count':    int(y_true.sum()),
    'metrics': {
        'precision': round(precision, 4),
        'recall':    round(recall, 4),
        'f1':        round(f1, 4),
        'roc_auc':   round(auc, 4),
    },
    'severity_thresholds': {
        'CRITICAL': 'iso_score < -0.20 OR qty_vs_mean > 8',
        'HIGH':     'iso_score < -0.10 OR qty_vs_mean > 5',
        'MEDIUM':   'flagged but below HIGH thresholds'
    },
    'anomaly_types_detectable': [
        'panic_bulk_order',
        'repeat_excess',
        'unusual_combo',
        'off_cycle_spike'
    ]
}

with open('/content/ML/models/anomaly_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# ── List files ────────────────────────────────────────────────────────────────
print('✅ All model files saved:')
print()
for fname in sorted(os.listdir('/content/ML/models/')):
    fpath = f'/content/ML/models/{fname}'
    size  = os.path.getsize(fpath)
    print(f'  {fname:45s}  {size/1024:.1f} KB')

# ── Download zip ──────────────────────────────────────────────────────────────
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/feature2_models.zip', 'w') as zf:
    for fname in os.listdir('/content/ML/models/'):
        zf.write(f'/content/ML/models/{fname}', fname)

print('\n📦 Downloading feature2_models.zip...')
files.download('/content/feature2_models.zip')

## FastAPI-Ready Inference Function

In [ ]:
# ── Full inference function — what FastAPI /detect-anomaly calls ──────────────

def detect_anomaly(pharmacy_id: str, sku_id: str, order_qty: int,
                   order_value: int, city_tier: int,
                   order_month: int, order_dow: int,
                   is_month_start: int, is_quarter_end: int,
                   iso_model, scaler, entity_stats,
                   use_gemini: bool = True):
    """
    Single function your FastAPI /detect-anomaly endpoint calls.

    Input  : order details
    Output : is_anomaly, severity, score, alert message
    """
    # Step 1: Get entity historical profile
    profile = entity_stats[
        (entity_stats['pharmacy_id'] == pharmacy_id) &
        (entity_stats['sku_id']      == sku_id)
    ]

    if len(profile) == 0:
        # New pharmacy with no history — flag for manual review
        return {
            'is_anomaly': True,
            'severity':   'MEDIUM',
            'reason':     'No order history found for this pharmacy-product pair',
            'alert':      'New pharmacy placing first order — recommend manual approval.',
            'iso_score':  None
        }

    p = profile.iloc[0]

    # Step 2: Compute ratio features
    qty_vs_mean   = order_qty / (p['mean_qty']   + 1e-6)
    qty_vs_median = order_qty / (p['median_qty'] + 1e-6)
    qty_vs_p95    = order_qty / (p['p95_qty']    + 1e-6)
    qty_vs_max    = order_qty / (p['max_qty']    + 1e-6)
    qty_zscore    = (order_qty - p['mean_qty']) / (p['std_qty'] + 1e-6)

    features = np.array([[
        qty_vs_mean, qty_vs_median, qty_vs_p95, qty_vs_max, qty_zscore,
        np.log1p(order_value), np.log1p(order_qty),
        np.log1p(p['mean_qty']), p['cv'], np.log1p(p['p95_qty']),
        p['total_orders'],
        p['mean_qty'], p['std_qty'], p['p95_qty'],
        order_month, order_dow, is_month_start, is_quarter_end
    ]])

    # Step 3: Scale and predict
    features_scaled = scaler.transform(features)
    raw_pred        = iso_model.predict(features_scaled)[0]
    iso_score       = float(iso_model.decision_function(features_scaled)[0])
    is_anomaly      = (raw_pred == -1)

    severity = classify_severity(iso_score, qty_vs_mean)

    # Step 4: Gemini alert (only if anomaly detected)
    alert = None
    if is_anomaly and use_gemini:
        row_dict = {
            'pharmacy_name': pharmacy_id,
            'city':          'India',
            'city_tier':     city_tier,
            'product_name':  sku_id,
            'order_qty':     order_qty,
            'mean_qty':      p['mean_qty'],
            'qty_vs_mean':   qty_vs_mean,
            'qty_zscore':    qty_zscore,
            'iso_score':     iso_score,
        }
        alert, _ = generate_admin_alert(pd.Series(row_dict))
    elif is_anomaly:
        alert = (
            f'Order of {order_qty} units is {qty_vs_mean:.1f}x the normal average '
            f'({p["mean_qty"]:.0f} units). Recommend manual review.'
        )

    return {
        'pharmacy_id':   pharmacy_id,
        'sku_id':        sku_id,
        'order_qty':     order_qty,
        'normal_avg':    round(float(p['mean_qty']), 1),
        'qty_vs_mean':   round(qty_vs_mean, 2),
        'qty_zscore':    round(qty_zscore, 2),
        'is_anomaly':    bool(is_anomaly),
        'iso_score':     round(iso_score, 4),
        'severity':      severity if is_anomaly else 'NONE',
        'alert':         alert,
    }


# ── Test the function ─────────────────────────────────────────────────────────
print('🔬 Testing inference function...\n')

# Test 1: Normal order
r1 = detect_anomaly(
    'PH_01', 'SKU_004', order_qty=25, order_value=25*560,
    city_tier=1, order_month=3, order_dow=0,
    is_month_start=1, is_quarter_end=1,
    iso_model=iso_model, scaler=scaler, entity_stats=entity_stats,
    use_gemini=False
)
print('TEST 1 — Normal order (qty=25):')
for k, v in r1.items(): print(f'  {k:20s}: {v}')

print()

# Test 2: Suspicious bulk order
r2 = detect_anomaly(
    'PH_01', 'SKU_004', order_qty=900, order_value=900*560,
    city_tier=1, order_month=3, order_dow=0,
    is_month_start=1, is_quarter_end=1,
    iso_model=iso_model, scaler=scaler, entity_stats=entity_stats,
    use_gemini=True
)
print('TEST 2 — Suspicious bulk order (qty=900):')
for k, v in r2.items(): print(f'  {k:20s}: {v}')

In [ ]:
# ── FastAPI endpoint code (copy to ML/api/main.py) ───────────────────────────
FASTAPI_CODE = '''
# Add this to ML/api/main.py alongside Feature 1 endpoint

import pandas as pd

# Load at startup (alongside demand_forecaster.pkl)
anomaly_model  = joblib.load("models/anomaly_detector.pkl")
anomaly_scaler = joblib.load("models/anomaly_scaler.pkl")
entity_stats   = pd.read_csv("models/entity_stats.csv")
anom_meta      = json.load(open("models/anomaly_metadata.json"))


class AnomalyRequest(BaseModel):
    pharmacy_id:    str
    sku_id:         str
    order_qty:      int
    order_value:    int
    city_tier:      int
    order_month:    int
    order_dow:      int
    is_month_start: int
    is_quarter_end: int
    pharmacy_name:  str
    product_name:   str
    city:           str


@app.post("/detect-anomaly")
async def detect_anomaly_endpoint(req: AnomalyRequest):
    profile = entity_stats[
        (entity_stats["pharmacy_id"] == req.pharmacy_id) &
        (entity_stats["sku_id"]      == req.sku_id)
    ]
    if len(profile) == 0:
        return {"is_anomaly": True, "severity": "MEDIUM",
                "alert": "No order history — manual review recommended."}

    p = profile.iloc[0]
    qty_vs_mean   = req.order_qty / (p["mean_qty"] + 1e-6)
    qty_vs_median = req.order_qty / (p["median_qty"] + 1e-6)
    qty_vs_p95    = req.order_qty / (p["p95_qty"] + 1e-6)
    qty_vs_max    = req.order_qty / (p["max_qty"] + 1e-6)
    qty_zscore    = (req.order_qty - p["mean_qty"]) / (p["std_qty"] + 1e-6)

    features = np.array([[
        qty_vs_mean, qty_vs_median, qty_vs_p95, qty_vs_max, qty_zscore,
        np.log1p(req.order_value), np.log1p(req.order_qty),
        np.log1p(p["mean_qty"]), p["cv"], np.log1p(p["p95_qty"]),
        p["total_orders"], p["mean_qty"], p["std_qty"], p["p95_qty"],
        req.order_month, req.order_dow, req.is_month_start, req.is_quarter_end
    ]])

    scaled      = anomaly_scaler.transform(features)
    iso_score   = float(anomaly_model.decision_function(scaled)[0])
    is_anomaly  = bool(anomaly_model.predict(scaled)[0] == -1)

    if iso_score < -0.20 or qty_vs_mean > 8: severity = "CRITICAL"
    elif iso_score < -0.10 or qty_vs_mean > 5: severity = "HIGH"
    else: severity = "MEDIUM"

    alert = None
    if is_anomaly:
        prompt = f"""
        Swasthya admin alert. 2-sentence summary. No markdown.
        Pharmacy: {req.pharmacy_name}, {req.city}
        Product: {req.product_name}
        Ordered: {req.order_qty} units (normal avg: {p["mean_qty"]:.0f})
        Ratio: {qty_vs_mean:.1f}x normal | Z-score: {qty_zscore:.1f} | Severity: {severity}
        End with recommended action.
        """
        alert = gemini.generate_content(prompt).text

    return {
        "pharmacy_id": req.pharmacy_id,
        "sku_id":      req.sku_id,
        "order_qty":   req.order_qty,
        "normal_avg":  round(float(p["mean_qty"]), 1),
        "qty_vs_mean": round(qty_vs_mean, 2),
        "is_anomaly":  is_anomaly,
        "iso_score":   round(iso_score, 4),
        "severity":    severity if is_anomaly else "NONE",
        "alert":       alert,
    }
'''

print(FASTAPI_CODE)
print('\n✅ Copy the above into ML/api/main.py (add below Feature 1 endpoint)')

---
## Feature 2 : Anomaly Detection

### Files saved:
```
ML/
└── models/
    ├── anomaly_detector.pkl          ← Isolation Forest model
    ├── anomaly_scaler.pkl            ← StandardScaler
    ├── entity_stats.csv              ← per pharmacy+product profiles
    ├── anomaly_metadata.json         ← metrics + config for FastAPI
    ├── flagged_orders.csv            ← all flagged orders for admin
    ├── anomaly_eda.png               ← for report
    ├── anomaly_evaluation.png        ← for report (confusion matrix, ROC)
    ├── threshold_tuning.png          ← for report
    ├── anomaly_feature_importance.png← for report
    └── admin_dashboard.png           ← for report
```
